# 🚀 Day 1: Guided Lab - Your First LLM Interactions with Gemini

Welcome to your first hands-on session with Large Language Models!

## Learning Objectives
By the end of this lab, you will be able to:
- ✅ Set up and use the Gemini API in Google Colab
- ✅ Make API calls and understand the response structure
- ✅ Experiment with parameters like temperature and max tokens
- ✅ Build multi-turn conversations
- ✅ Create a simple business application

## Time: 90 minutes

---

## 🎯 Ground Rules for Working with LLMs

Keep these principles in mind throughout the course:

1. **If unsure, say so** - LLMs should acknowledge uncertainty, not make things up
2. **Output format matters** - Always specify the format you need (especially JSON)
3. **Don't invent facts** - LLMs should only use information provided in the input
4. **Verify important outputs** - Always spot-check critical results

---

## Part 1: Setup (15 minutes)

### 1.1 Install the Gemini SDK

Run the cell below to install the Google GenAI package.

> ⚠️ You may see a dependency warning about `google-auth` - this is safe to ignore.

In [ ]:
!pip -q install -U google-genai

### 1.2 Load Your API Key

Make sure you've added your `GEMINI_API_KEY` to Colab Secrets (🔑 icon in the left sidebar).

The secret name must be exactly: `GEMINI_API_KEY`

In [ ]:
import os
import json
import time
from datetime import datetime, timezone
from google import genai
from google.genai import types

# --- Load API key ---
# Option A (recommended): Colab Secrets
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

# Option B (fallback): hidden paste
if not os.environ.get("GEMINI_API_KEY"):
    import getpass
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Paste your GEMINI_API_KEY (input hidden): ")

# Verify
if os.environ.get("GEMINI_API_KEY"):
    print("✅ API key loaded successfully!")
else:
    print("❌ API key not found. Check your Colab Secrets.")

### 1.3 Initialize Client and Helper Functions

We'll set up:
1. The Gemini client
2. A **prompt log** to track all our API calls (useful for learning!)
3. Helper functions we'll use throughout the course

In [ ]:
# Initialize the client
client = genai.Client()

# Model configuration
MODEL_ID = "gemini-2.0-flash-lite"  # Fast and free-tier friendly

# --- Prompt Logging ---
# This helps you track what you tried and what worked
PROMPT_LOG = []

def _now():
    """Get current timestamp."""
    return datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')

def generate(prompt, temperature=0.7, max_tokens=500, log=True):
    """
    Generate text using Gemini with customizable parameters.
    Automatically logs all calls for later review.

    Args:
        prompt (str): The input prompt
        temperature (float): Creativity level (0.0 to 2.0)
        max_tokens (int): Maximum output length
        log (bool): Whether to log this call

    Returns:
        str: The generated text
    """
    t0 = time.time()

    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=temperature,
            max_output_tokens=max_tokens
        )
    )

    text = response.text or ""
    latency = time.time() - t0

    # Log the call
    if log:
        PROMPT_LOG.append({
            "ts": _now(),
            "model": MODEL_ID,
            "temperature": temperature,
            "max_tokens": max_tokens,
            "prompt": prompt,
            "output": text,
            "latency_s": round(latency, 3)
        })

    return text

def try_parse_json(text):
    """
    Attempt to parse JSON from LLM output.
    Returns (success: bool, parsed_data: dict/list or None)
    """
    # Try direct parse
    try:
        return True, json.loads(text)
    except json.JSONDecodeError:
        pass

    # Try to extract JSON from markdown code blocks
    import re
    json_match = re.search(r'```(?:json)?\s*([\s\S]*?)```', text)
    if json_match:
        try:
            return True, json.loads(json_match.group(1))
        except json.JSONDecodeError:
            pass

    # Try to find JSON object/array in text
    json_match = re.search(r'[\[\{][\s\S]*[\]\}]', text)
    if json_match:
        try:
            return True, json.loads(json_match.group())
        except json.JSONDecodeError:
            pass

    return False, None

def show_log(n=3):
    """Display the last n entries from the prompt log."""
    for row in PROMPT_LOG[-n:]:
        print("=" * 70)
        print(f"Time: {row['ts']} | Temp: {row['temperature']} | Latency: {row['latency_s']}s")
        print("--- PROMPT (first 500 chars) ---")
        print(row["prompt"][:500])
        print("--- OUTPUT (first 500 chars) ---")
        print(row["output"][:500])

print(f"✅ Client initialized! Using model: {MODEL_ID}")
print(f"📝 Prompt logging enabled. Use show_log() to review your calls.")

### 1.4 Test Your Connection

Let's make sure everything works with a simple test.

In [ ]:
# Simple test call
response = generate(
    "Say 'Hello, I am ready to learn!' in a friendly way.",
    temperature=0.7
)

print(response)

🎉 **If you see a friendly greeting above, you're all set!**

---

## Part 2: Basic Text Generation (15 minutes)

### 2.1 Your First Business Prompt

Let's generate a product description - a common business use case.

In [ ]:
# Simple product description
prompt = "Write a product description for wireless noise-canceling headphones in 2-3 sentences."

response = generate(prompt)

print("Generated Description:")
print(response)

### 2.2 Run It Multiple Times

Run the cell below multiple times. Notice how the output varies each time!

This is because LLMs are **probabilistic** - they don't always give the same answer.

In [ ]:
# Run this cell 3-4 times and observe the different outputs
response = generate(
    "Suggest a creative name for a new coffee shop.",
    temperature=0.9  # Higher temperature = more variety
)

print(response)

### 2.3 Understanding the Response Object

Let's look at more details from the API response, including token usage.

In [ ]:
# Make a call and examine the full response
full_response = client.models.generate_content(
    model=MODEL_ID,
    contents="What are three benefits of cloud computing for small businesses?"
)

# The main text output
print("=== Response Text ===")
print(full_response.text)

# Usage statistics (tokens used)
print("\n=== Usage Statistics ===")
print(f"Input tokens: {full_response.usage_metadata.prompt_token_count}")
print(f"Output tokens: {full_response.usage_metadata.candidates_token_count}")
print(f"Total tokens: {full_response.usage_metadata.total_token_count}")

💡 **Why tokens matter:**
- API pricing is based on tokens
- There are limits on input and output tokens
- 1 token ≈ 4 characters or ¾ of a word in English

---

## Part 3: Exploring Parameters (15 minutes)

### 3.1 Temperature: Controlling Creativity

**Temperature** controls how "creative" or "random" the output is:
- `temperature=0.0` → Deterministic, consistent outputs
- `temperature=1.0` → More creative, varied outputs
- `temperature=2.0` → Very random (often too chaotic)

Let's see this in action!

In [ ]:
prompt = "Suggest a name for a new smartphone app that helps people track their fitness goals."

print("=" * 50)
print("TEMPERATURE = 0.0 (Deterministic)")
print("Running 3 times - outputs should be very similar:")
print("=" * 50)

for i in range(3):
    response = generate(prompt, temperature=0.0)
    print(f"Run {i+1}: {response.strip()[:300]}")

In [ ]:
print("=" * 50)
print("TEMPERATURE = 1.0 (Creative)")
print("Running 3 times - outputs should be different:")
print("=" * 50)

for i in range(3):
    response = generate(prompt, temperature=1.0)
    print(f"Run {i+1}: {response.strip()[:300]}")

### 💭 Discussion Question

**When would you use low temperature vs high temperature in a business context?**

- Low temperature (0.0-0.3): _______________________
- High temperature (0.7-1.0): _______________________

### 3.2 Max Output Tokens: Controlling Length

You can limit how long the response should be.

In [ ]:
prompt = "Explain the concept of machine learning to a business executive."

# Short response (max 50 tokens)
print("=== SHORT (max 50 tokens) ===")
response_short = generate(prompt, max_tokens=50)
print(response_short)

# Check actual length from log
print(f"\n(Check the log for actual token count)")

In [ ]:
# Longer response (max 200 tokens)
print("=== LONGER (max 200 tokens) ===")
response_long = generate(prompt, max_tokens=200)
print(response_long)

### 3.3 Review Your Prompt Log

Let's see what we've tried so far!

In [ ]:
# See how many calls we've made
print(f"Total API calls so far: {len(PROMPT_LOG)}")

# Show the last 3 calls
show_log(3)

---

## Part 4: Multi-turn Conversations (15 minutes)

LLMs can maintain context across multiple exchanges, just like a real conversation.

### 4.1 Starting a Chat Session

In [ ]:
# Create a chat session
chat = client.chats.create(model=MODEL_ID)

# First message
response1 = chat.send_message("Hi! I'm looking for a laptop for video editing. What should I consider?")
print("User: Hi! I'm looking for a laptop for video editing. What should I consider?")
print(f"\nAssistant: {response1.text}")

In [ ]:
# Follow-up message - the model remembers the context!
response2 = chat.send_message("My budget is around $1500. Any specific recommendations?")
print("User: My budget is around $1500. Any specific recommendations?")
print(f"\nAssistant: {response2.text}")

In [ ]:
# Another follow-up
response3 = chat.send_message("What about battery life? I travel a lot.")
print("User: What about battery life? I travel a lot.")
print(f"\nAssistant: {response3.text}")

### 4.2 View Chat History

You can access the full conversation history.

In [ ]:
# View the conversation history
print("=== Full Chat History ===")
for i, message in enumerate(chat._curated_history):
    role = "👤 User" if message.role == "user" else "🤖 Assistant"
    text = message.parts[0].text if message.parts else ""
    preview = text[:150] + "..." if len(text) > 150 else text
    print(f"\n{role}: {preview}")

### 💭 Discussion Question

**How might multi-turn conversations be useful in a business application?**

Think of 2-3 examples: _______________________

---

## Part 5: Business Application - Email Response Generator (15 minutes)

Let's build something practical: a customer service email response generator.

### 5.1 Basic Email Response

In [ ]:
def generate_email_response(customer_email, company_name="TechCorp", tone="professional"):
    """
    Generate a customer service response to an email.

    Args:
        customer_email (str): The customer's email content
        company_name (str): Your company name
        tone (str): Response tone (professional, friendly, formal)

    Returns:
        str: Generated email response
    """
    prompt = f"""You are a customer service representative for {company_name}.
Write a {tone} response to this customer email.

Customer Email:
---
{customer_email}
---

Requirements:
- Acknowledge their concern empathetically
- Provide a clear solution or next steps
- Keep the response under 150 words
- End with a positive note
- Do not invent facts not present in the email

Response:"""

    return generate(prompt, temperature=0.3)  # Low temperature for consistency


# Test with a sample email
sample_email = """
Hi,

I ordered a laptop last week (Order #12345) and it arrived today with a cracked screen.
This is really frustrating as I needed it for an important presentation tomorrow.

What can you do about this?

Thanks,
John
"""

response = generate_email_response(sample_email)
print("Generated Response:")
print(response)

### 5.2 Try Different Tones

In [ ]:
# Same email, different tones
print("=== PROFESSIONAL TONE ===")
print(generate_email_response(sample_email, tone="professional"))

print("\n" + "="*50 + "\n")

print("=== FRIENDLY TONE ===")
print(generate_email_response(sample_email, tone="friendly and warm"))

### 5.3 Meeting Summary Generator

Another common business use case: summarizing meeting notes.

In [ ]:
def summarize_meeting(transcript):
    """
    Summarize a meeting transcript into key points and action items.
    """
    prompt = f"""Summarize the following meeting transcript.

Provide:
1. A 2-3 sentence summary
2. Key decisions made (bullet points)
3. Action items with owners (if mentioned)

Rules:
- Only include information from the transcript
- If owner/deadline not mentioned, write "Not specified"
- Be concise

Meeting Transcript:
---
{transcript}
---

Summary:"""

    return generate(prompt, temperature=0.2)


# Sample meeting transcript
meeting_transcript = """
Sarah: Okay everyone, let's discuss the Q2 marketing budget.

Mike: I think we should increase social media spending by 20%. Our Instagram campaigns
have been performing really well.

Sarah: Good point. What about the trade show in March?

Lisa: The trade show costs about $15,000. I think it's worth it for the leads we get.
Last year we got 50 qualified leads from it.

Mike: Agreed. Let's keep the trade show budget.

Sarah: Alright, so we'll increase social media by 20% and maintain trade show spending.
Mike, can you prepare a detailed social media plan by Friday?

Mike: Sure, I'll have it ready.

Sarah: Lisa, please confirm our trade show registration.

Lisa: Will do, I'll handle it this week.

Sarah: Great, let's reconvene next Tuesday to finalize everything.
"""

summary = summarize_meeting(meeting_transcript)
print("Meeting Summary:")
print(summary)

### 5.4 Testing JSON Output (Preview of Day 2)

Sometimes you need structured data, not just text. Let's try getting JSON output.

In [ ]:
# Request JSON output
json_prompt = """Extract action items from this meeting note. Return JSON ONLY.

Meeting note: "Mike will prepare social media plan by Friday. Lisa will confirm trade show registration this week."

Schema:
[
  {"owner": "name", "task": "description", "deadline": "when"}
]

JSON:"""

response = generate(json_prompt, temperature=0.0)
print("Raw output:")
print(response)

# Try to parse it
success, parsed = try_parse_json(response)
print(f"\nValid JSON? {success}")
if success:
    print("Parsed data:")
    for item in parsed:
        print(f"  - {item}")

---

## Part 6: Wrap-up (15 minutes)

### 6.1 Export Your Prompt Log

Save a record of everything you tried today!

In [ ]:
import pandas as pd

# Convert log to DataFrame
df = pd.DataFrame(PROMPT_LOG)

# Save to CSV
df.to_csv("day1_guided_lab_prompt_log.csv", index=False)
print(f"✅ Saved {len(df)} prompts to: day1_guided_lab_prompt_log.csv")

# Show summary
print(f"\n📊 Session Summary:")
print(f"   Total API calls: {len(df)}")
print(f"   Average latency: {df['latency_s'].mean():.2f}s")
print(f"   Temperature range: {df['temperature'].min()} - {df['temperature'].max()}")

### 6.2 Key Takeaways

**What we learned today:**

1. **API Basics**: You can interact with LLMs through simple API calls
2. **Parameters Matter**: Temperature and max tokens control output behavior
3. **Conversations**: LLMs can maintain context across multiple turns
4. **Business Value**: Even simple prompts can automate useful tasks
5. **Logging Helps**: Tracking your prompts helps you learn what works

### 6.3 Reflection Questions

Before moving to the independent lab, discuss with a partner:

1. What surprised you most about working with LLMs?
2. What limitations did you notice?
3. What business process in your experience could benefit from this technology?

---

## 🎯 Ready for the Independent Lab!

In the next session, you'll work independently (or with a partner) to:
- Build a content generation suite
- Create text transformation tools
- Develop an interactive Q&A system

**Save this notebook** - you'll reuse the helper functions!